# Inkengine CUDA RORET engine-level logits parity (M17)

Goal: on a Kaggle 2xT4 GPU runner, measure ENGINE-LEVEL raw F32 logits
parity of the union-resident MoE (RORET) wiring, RORET off vs on, for both
cpu_share = 0.0 and cpu_share = 1.0, on the deterministic synthetic GGUF.
This is evidence for GitHub issue #4.

What is compared, byte for byte:
- llama-debug --save-logits raw F32 .bin files: final raw F32 logits
- llama-debug token-id .bin files
- llama-cli generated text bytes (-o), as a cross-check

Scope limits:
- Synthetic GGUF only. No real weights.
- The logits files are raw F32 FINAL logits from llama-debug, NOT
  intermediate per-pass MoE tensors.
- RORET_PASS rows come from the PAIRED llama-cli run, not llama-debug.
  They are count evidence that the copy path ran, not timing.
- Both binaries are run with --verbose; llama-debug emits no INFO lines
  (including the RORET hook line) without it.

NO SPEED CLAIM. This notebook makes no speed or throughput claim.
Any future speed number needs CycleAudit claim <= ci_low with repeats >= 20 and loops = 3.

Fail-closed: any failed stage skips later stages and the final summary prints NO_GO.

In [ ]:
import os, sys, json, re, shutil, hashlib, subprocess, filecmp
from pathlib import Path

WORK = "/kaggle/working"
REPO = WORK + "/inkengine-llama"
GGUF = WORK + "/synthetic.gguf"
HOOK_LINE = "RORET union-resident MoE bookkeeping enabled"
SPEED_GATE = "CycleAudit claim <= ci_low with repeats >= 20 and loops = 3"

def run_logged(cmd, log_path, cwd=None, env=None, timeout=None):
    # timeout is ONLY for the eight short binary runs. The CUDA build needs
    # longer than 300 s and must not inherit the per-run timeout.
    try:
        with open(log_path, "w") as f:
            p = subprocess.run(cmd, cwd=cwd, env=env, stdout=f,
                               stderr=subprocess.STDOUT, timeout=timeout)
        return p.returncode
    except subprocess.TimeoutExpired:
        with open(log_path, "a") as f:
            f.write("\nTIMEOUT: command exceeded " + str(timeout) + " seconds\n")
        return 124

def tail(path, n=40):
    try:
        lines = open(path, errors="replace").read().splitlines()
    except OSError:
        return ""
    return "\n".join(lines[-n:])

def sha256_file(path):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(1 << 20), b""):
            h.update(chunk)
    return h.hexdigest()

def differing_bytes(p1, p2):
    b1 = open(p1, "rb").read()
    b2 = open(p2, "rb").read()
    return sum(a != b for a, b in zip(b1, b2)) + abs(len(b1) - len(b2))

def bin_names(d):
    if not os.path.isdir(d):
        return []
    return sorted(p.name for p in Path(d).glob("*.bin") if p.is_file() and p.stat().st_size > 0)

print("python", sys.version.split()[0])

HAVE_NVCC = shutil.which("nvcc") is not None
if HAVE_NVCC:
    r = subprocess.run(["nvcc", "--version"], capture_output=True, text=True)
    nvcc_version = (r.stdout or r.stderr).strip()
    print(nvcc_version)
else:
    nvcc_version = "BLOCKER: no nvcc"
    print("BLOCKER: no nvcc")

r = subprocess.run(["nvidia-smi"], capture_output=True, text=True)
print("nvidia-smi rc", r.returncode)
print((r.stdout or r.stderr)[:2000])

try:
    import torch
    names = [torch.cuda.get_device_name(i) for i in range(torch.cuda.device_count())]
    print("torch", torch.__version__, "cuda", getattr(torch.version, "cuda", "n/a"))
    print("device_count", torch.cuda.device_count(), names)
except Exception as e:
    print("torch unavailable:", type(e).__name__)

summary = {
    "head_sha": "",
    "nvcc_version": nvcc_version,
    "nvcc_available": HAVE_NVCC,
    "build_ok": False,
    "configure_command": [],
    "configure_exit": None,
    "build_exit": None,
    "configure_log_tail": "",
    "build_log_tail": "",
    "fixture_sha256": "",
    "runs": {},
    "logits_parity_verdict": "NOT_RUN",
    "text_parity_verdict": "NOT_RUN",
    "engine_verdict": "NOT_RUN",
    "speed_claim": "NO SPEED CLAIM",
    "speed_gate": SPEED_GATE,
    "blocker": "",
    "overall_verdict": "NO_GO",
}
SHARES = ("0.0", "1.0")
for share in SHARES:
    summary["runs"][share] = {
        "off_exit": None,
        "on_exit": None,
        "off_hook_active": None,
        "on_hook_active": None,
        "off_hook_active_cli": None,
        "on_hook_active_cli": None,
        "off_hook_active_debug": None,
        "on_hook_active_debug": None,
        "logits_files": {},
        "token_files": {},
        "text_sha": {},
        "logits_differing_bytes": None,
        "token_differing_bytes": None,
        "text_differing_bytes": None,
        "roret_pass_rows": {},
        "roret_pass_source": {},
        "assert_lines": {},
        "share_verdict": "NOT_RUN",
        "share_text_verdict": "NOT_RUN",
    }

PIPELINE_OK = HAVE_NVCC
if not PIPELINE_OK:
    summary["blocker"] = "no nvcc"
    print("BLOCKER: no nvcc; pipeline stopped. Summary cell will still run.")


In [ ]:
if not PIPELINE_OK:
    print("SKIP: pipeline blocked")
else:
    if not os.path.isdir(REPO):
        cmd = ["git", "clone", "-b", "main",
               "https://github.com/10xdev4u-alt/inkengine-llama.git", REPO]
        print("RUN", " ".join(cmd))
        rc = subprocess.run(cmd).returncode
        print("clone exit", rc)
        if rc != 0:
            PIPELINE_OK = False
            summary["blocker"] = "git clone failed"
    else:
        print("clone: existing dir, skip", REPO)
    if PIPELINE_OK:
        r = subprocess.run(["git", "rev-parse", "HEAD"], cwd=REPO,
                           capture_output=True, text=True)
        summary["head_sha"] = r.stdout.strip()
        print("HEAD", summary["head_sha"])
        if not summary["head_sha"]:
            PIPELINE_OK = False
            summary["blocker"] = "git rev-parse HEAD failed"


In [ ]:
if not PIPELINE_OK:
    print("SKIP: pipeline blocked")
else:
    cfg = ["cmake", "-S", ".", "-B", "build", "-DGGML_CUDA=ON",
           "-DCMAKE_CUDA_ARCHITECTURES=75", "-DLLAMA_BUILD_TESTS=OFF",
           "-DLLAMA_CURL=OFF", "-DGGML_CUDA_NO_VMM=ON"]
    summary["configure_command"] = cfg
    print("RUN", " ".join(cfg))
    rc1 = run_logged(cfg, WORK + "/cmake_configure.log", cwd=REPO)
    summary["configure_exit"] = rc1
    summary["configure_log_tail"] = tail(WORK + "/cmake_configure.log", n=20)
    print("configure exit", rc1)
    print("configure log tail:")
    print(summary["configure_log_tail"])
    if rc1 != 0:
        print(tail(WORK + "/cmake_configure.log", n=80))

    bld = ["cmake", "--build", "build", "--target", "llama-cli", "llama-debug", "-j4"]
    print("RUN", " ".join(bld))
    rc2 = run_logged(bld, WORK + "/cmake_build.log", cwd=REPO)
    summary["build_exit"] = rc2
    summary["build_log_tail"] = tail(WORK + "/cmake_build.log", n=20)
    print("build exit", rc2)
    if rc2 != 0:
        print(summary["build_log_tail"])

    summary["build_ok"] = (rc1 == 0 and rc2 == 0)
    if summary["build_ok"]:
        for b in ("llama-cli", "llama-debug"):
            if not os.path.isfile(REPO + "/build/bin/" + b):
                summary["build_ok"] = False
                summary["blocker"] = "missing binary " + b
                print("BLOCKER: missing binary", b)
    PIPELINE_OK = summary["build_ok"]
    print("build_ok", summary["build_ok"])
    if not PIPELINE_OK:
        print("BLOCKER: CUDA build failed; later stages skipped")


In [ ]:
if not PIPELINE_OK:
    print("SKIP: pipeline blocked")
else:
    gen = REPO + "/tools/audit/make_synthetic_gguf.py"
    gen_env = dict(os.environ)
    gen_env["PYTHONPATH"] = REPO + "/gguf-py"
    cmd = [sys.executable, gen, "--out", GGUF]
    print("RUN", " ".join(cmd))
    rc = run_logged(cmd, WORK + "/make_gguf.log", cwd=REPO, env=gen_env)
    print("generator exit", rc)
    if rc != 0:
        print(tail(WORK + "/make_gguf.log"))
    if rc == 0 and os.path.isfile(GGUF) and os.path.getsize(GGUF) > 0:
        summary["fixture_sha256"] = sha256_file(GGUF)
        print("fixture_sha256", summary["fixture_sha256"])
        print("(informational only; not a gate)")
    else:
        summary["blocker"] = "GGUF generation failed or file missing/empty"
        PIPELINE_OK = False
        print("BLOCKER: GGUF generation failed or file missing/empty")


In [ ]:
if not PIPELINE_OK:
    print("SKIP: pipeline blocked")
else:
    cli = REPO + "/build/bin/llama-cli"
    dbg = REPO + "/build/bin/llama-debug"
    # --verbose is in base_args for BOTH binaries. Without it llama-debug
    # emits no INFO lines, including the RORET hook line.
    base_args = ["-m", GGUF, "-p", "Hello my name is", "-s", "0", "-n", "16",
                 "-t", "1", "-ngl", "99", "-cmoe", "--moe-cache-mib", "16",
                 "--temp", "0", "--verbose"]
    rx = re.compile(r"RORET_PASS\s+il=(-?\d+)\s+pass=(-?\d+)\s+plans=(\d+)\s+"
                    r"hits=(\d+)\s+upload_ops=(\d+)\s+uploaded_bytes=(\d+)\s+"
                    r"residents=(\d+)")
    row_keys = ("il", "pass", "plans", "hits", "upload_ops", "uploaded_bytes", "residents")
    for share in SHARES:
        rec = summary["runs"][share]
        for tag in ("off", "on"):
            env = dict(os.environ)
            if tag == "off":
                env.pop("LLAMA_RORET", None)
            else:
                env["LLAMA_RORET"] = "1"
            stem = share + "-" + tag
            txt = WORK + "/" + stem + "-cli.txt"
            cli_log = WORK + "/" + stem + "-cli.log"
            dlog = WORK + "/" + stem + "-debug.log"
            odir = WORK + "/" + stem + "-logits"
            for stale in (txt, cli_log, dlog):
                if os.path.isfile(stale):
                    os.remove(stale)
            if os.path.isdir(odir):
                shutil.rmtree(odir)

            cmd_cli = [cli] + base_args + ["--moe-cache-cpu-share", share, "-o", txt]
            print("RUN", tag, " ".join(cmd_cli))
            rc_cli = run_logged(cmd_cli, cli_log, env=env, timeout=300)
            cmd_dbg = [dbg] + base_args + ["--moe-cache-cpu-share", share,
                                          "--save-logits", "--logits-output-dir", odir]
            print("RUN", tag, " ".join(cmd_dbg))
            rc_dbg = run_logged(cmd_dbg, dlog, env=env, timeout=300)
            rec[tag + "_exit"] = {"cli": rc_cli, "debug": rc_dbg}
            print(stem, "cli exit", rc_cli, "debug exit", rc_dbg)
            if rc_cli != 0:
                print(stem, "cli log tail:")
                print(tail(cli_log, n=20))
            if rc_dbg != 0:
                print(stem, "debug log tail:")
                print(tail(dlog, n=20))

            lines_cli = []
            lines_debug = []
            for lp, sink in ((cli_log, lines_cli), (dlog, lines_debug)):
                try:
                    sink.extend(open(lp, errors="replace").read().splitlines())
                except OSError:
                    pass
            hook_cli = any(HOOK_LINE in ln for ln in lines_cli)
            hook_debug = any(HOOK_LINE in ln for ln in lines_debug)
            rec[tag + "_hook_active_cli"] = hook_cli
            rec[tag + "_hook_active_debug"] = hook_debug
            # mirrors debug hook state; per-binary keys below are authoritative
            rec[tag + "_hook_active"] = hook_debug
            print(stem, "cli hook", "ACTIVE" if hook_cli else "ABSENT",
                  "debug hook", "ACTIVE" if hook_debug else "ABSENT")
            # RORET_PASS rows are printed by llama-cli with --verbose,
            # NOT by llama-debug. Keep this attribution explicit.
            rows = [dict(zip(row_keys, (int(v) for v in m)))
                    for m in rx.findall("\n".join(lines_cli))]
            rec["roret_pass_rows"][tag] = rows
            rec["roret_pass_source"][tag] = "llama-cli"
            print(stem, "RORET_PASS rows from llama-cli", len(rows))
            for row in rows[:20]:
                print("  " + " ".join(str(row[k]) for k in row_keys))
            asserts = [ln for ln in lines_cli + lines_debug
                       if "GGML_ASSERT" in ln or "Assertion" in ln]
            rec["assert_lines"][tag] = asserts[:20]
            print(stem, "assert lines", len(asserts))
            for ln in asserts[:20]:
                print("  " + ln)
            if os.path.isfile(txt) and os.path.getsize(txt) > 0:
                rec["text_sha"][tag] = sha256_file(txt)
            rec["logits_files"][tag] = [n for n in bin_names(odir) if not n.endswith("-tokens.bin")]
            rec["token_files"][tag] = [n for n in bin_names(odir) if n.endswith("-tokens.bin")]
            print(stem, "logits bins", rec["logits_files"][tag],
                  "token bins", rec["token_files"][tag])


In [ ]:
if not PIPELINE_OK:
    print("SKIP: pipeline blocked")
else:
    share_verdicts = []
    text_verdicts = []
    for share in SHARES:
        rec = summary["runs"][share]
        stem_off = WORK + "/" + share + "-off"
        stem_on = WORK + "/" + share + "-on"
        off_txt = stem_off + "-cli.txt"
        on_txt = stem_on + "-cli.txt"
        exits_ok = (rec["off_exit"]["cli"] == 0 and rec["off_exit"]["debug"] == 0 and
                    rec["on_exit"]["cli"] == 0 and rec["on_exit"]["debug"] == 0)
        text_files_ok = (os.path.isfile(off_txt) and os.path.isfile(on_txt) and
                         os.path.getsize(off_txt) > 0 and os.path.getsize(on_txt) > 0)
        if text_files_ok:
            dtext = differing_bytes(off_txt, on_txt)
            rec["text_differing_bytes"] = dtext
            print(share, "text cmp identical", filecmp.cmp(off_txt, on_txt, shallow=False),
                  "differing bytes", dtext)
        else:
            print(share, "text files missing or empty")

        off_logits = rec["logits_files"]["off"]
        on_logits = rec["logits_files"]["on"]
        off_tokens = rec["token_files"]["off"]
        on_tokens = rec["token_files"]["on"]
        sets_ok = (len(off_logits) > 0 and off_logits == on_logits and
                   len(off_tokens) > 0 and off_tokens == on_tokens)
        print(share, "logits file sets", off_logits, on_logits)
        print(share, "token file sets", off_tokens, on_tokens)
        if sets_ok:
            ld = max(differing_bytes(stem_off + "-logits/" + n, stem_on + "-logits/" + n)
                     for n in off_logits)
            td = max(differing_bytes(stem_off + "-logits/" + n, stem_on + "-logits/" + n)
                     for n in off_tokens)
            rec["logits_differing_bytes"] = ld
            rec["token_differing_bytes"] = td
            for n in off_logits:
                print(share, "logits", n, "identical",
                      filecmp.cmp(stem_off + "-logits/" + n, stem_on + "-logits/" + n, shallow=False))
            for n in off_tokens:
                print(share, "tokens", n, "identical",
                      filecmp.cmp(stem_off + "-logits/" + n, stem_on + "-logits/" + n, shallow=False))

        no_asserts = (not rec["assert_lines"]["off"] and not rec["assert_lines"]["on"])
        # Logits parity is valid ONLY if the llama-debug off run has the
        # RORET hook ABSENT and the llama-debug on run has it ACTIVE.
        debug_hook_gate_ok = (rec["off_hook_active_debug"] is False and
                              rec["on_hook_active_debug"] is True)
        gates = {
            "debug_hook_gate_ok": debug_hook_gate_ok,
            "exits_zero": exits_ok,
            "text_files_present": text_files_ok,
            "bin_sets_match_nonempty": sets_ok,
            "no_asserts": no_asserts,
            "text_diff_zero": rec["text_differing_bytes"] == 0,
            "logits_diff_zero": rec["logits_differing_bytes"] == 0,
            "token_diff_zero": rec["token_differing_bytes"] == 0,
        }
        failed = [k for k, v in gates.items() if not v]
        if not failed:
            rec["share_verdict"] = "LOGITS_PARITY: 0 differing bytes"
            rec["share_text_verdict"] = "TEXT_PARITY: 0 differing bytes"
        else:
            rec["share_verdict"] = "NO_GO: " + ",".join(failed)
            rec["share_text_verdict"] = "NO_GO: " + ",".join(failed)
            if summary["blocker"] == "":
                summary["blocker"] = "cpu_share=" + share + " gate failed: " + ",".join(failed)
        print(share, rec["share_verdict"])
        print(share, rec["share_text_verdict"])
        share_verdicts.append((share, rec["share_verdict"]))
        text_verdicts.append((share, rec["share_text_verdict"]))

    if all(v == "LOGITS_PARITY: 0 differing bytes" for _, v in share_verdicts):
        summary["logits_parity_verdict"] = "LOGITS_PARITY: 0 differing bytes (cpu_share=0.0 and 1.0)"
    else:
        summary["logits_parity_verdict"] = "NO_GO: " + "; ".join(s + "=" + v for s, v in share_verdicts)
    if all(v == "TEXT_PARITY: 0 differing bytes" for _, v in text_verdicts):
        summary["text_parity_verdict"] = "TEXT_PARITY: 0 differing bytes (cpu_share=0.0 and 1.0)"
    else:
        summary["text_parity_verdict"] = "NO_GO: " + "; ".join(s + "=" + v for s, v in text_verdicts)
    print(summary["logits_parity_verdict"])
    print(summary["text_parity_verdict"])
    print("Raw F32 logits bytes only; no intermediate per-pass MoE tensor parity.")


In [ ]:
if not PIPELINE_OK:
    summary["engine_verdict"] = "NO_GO: pipeline blocked before runs"
    print(summary["engine_verdict"])
else:
    all_rows = []
    off_hook_any = False
    on_hook_all = True
    rows_nonempty_all = True
    for share in SHARES:
        rec = summary["runs"][share]
        all_rows.extend(rec["roret_pass_rows"].get("on", []))
        if rec["off_hook_active_cli"]:
            off_hook_any = True
        if not rec["on_hook_active_cli"]:
            on_hook_all = False
        if not rec["roret_pass_rows"].get("on"):
            rows_nonempty_all = False
    summary["all_roret_pass_rows"] = all_rows
    print("total RORET_PASS rows from llama-cli logs across both shares", len(all_rows))
    for row in all_rows:
        print("  " + " ".join(str(row[k]) for k in row_keys))
    if on_hook_all and not off_hook_any and rows_nonempty_all and len(all_rows) > 0:
        summary["engine_verdict"] = "ENGINE_COPY_PATH_EXERCISED: YES"
    else:
        reasons = []
        if not on_hook_all:
            reasons.append("llama-cli on hook not active in every share")
        if off_hook_any:
            reasons.append("llama-cli off hook active in at least one share")
        if not rows_nonempty_all or len(all_rows) == 0:
            reasons.append("RORET_PASS table empty")
        summary["engine_verdict"] = "NO_GO: " + "; ".join(reasons)
    print(summary["engine_verdict"])

good = (summary["logits_parity_verdict"].startswith("LOGITS_PARITY:") and
        summary["text_parity_verdict"].startswith("TEXT_PARITY:") and
        summary["engine_verdict"].startswith("ENGINE_COPY_PATH_EXERCISED:") and
        summary["blocker"] == "")
summary["overall_verdict"] = "PASS" if good else "NO_GO"
print("overall_verdict", summary["overall_verdict"])
print("Scope: raw F32 FINAL logits bytes from llama-debug on ONE deterministic synthetic GGUF only.")
print("No real weights. No intermediate per-pass MoE tensor parity.")
print("RORET_PASS rows are from the paired llama-cli logs, not llama-debug.")
print("RORET_PASS rows are count evidence, not timing.")
print("NO SPEED CLAIM")
print("Any speed number needs " + SPEED_GATE + ".")


In [ ]:
out_path = WORK + "/engine_parity_summary.json"
with open(out_path, "w") as f:
    json.dump(summary, f, indent=2)
print(json.dumps(summary, indent=2))
print("summary written to", out_path)
if summary.get("overall_verdict") != "PASS":
    print("NO_GO: see blocker and per-share verdicts above.")
